# 네이버 뉴스 통합 수집기 — 기사 + 댓글 + 댓글 조치유형

## 실행 순서

`[0]` 설정 → `[1]` 링크 → `[2]` 기사 → `[3]` 필터 → `[4]` 댓글
→ `[5]` HTML 검증(선택) → `[6]` 저장

## [0] 설정 · import

**설정 블록만 고치면 된다.** 아래 셀들은 손댈 필요 없다.

```
pip install requests beautifulsoup4 pandas openpyxl tqdm lxml
# ACTION_HTML 을 쓸 때만: pip install selenium chromedriver-autoinstaller
```

In [ ]:
import os, re, csv, json, time, html, hashlib, unicodedata, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from bs4 import BeautifulSoup

try:
    from tqdm.auto import tqdm
except ImportError:                                    # tqdm 없어도 동작
    def tqdm(x, **k): return x

# ══════════════════════════════════════════════════════════════════════════════
#  설정  ★ 이 블록만 수정하세요 ★
# ══════════════════════════════════════════════════════════════════════════════
OUT_DIR    = r"C:\Users\com\Desktop\민주당전당대회결과" #데이터 저장 경로

QUERIES    = ["민주당 전당대회", "김민석", "정청래", "송영길"] #수집을 원하는 키워드를 입력. 중복될 경우 코드에서 정제해줌.
START_DATE = "20260817"
END_DATE   = "20260817"

# ── 수집 대상 언론사 ─────────────────────────────────────────────────────────
# 아래 TARGET 을 바꾸면 된다. 전체 목록은 MEDIA_GROUPS 에 있다.
#   이름으로  : TARGET = ["한겨레", "조선일보"]
#   그룹으로  : TARGET = "종합일간지"        (또는 ["종합일간지", "경제지"])
#   전부      : TARGET = "ALL"
TARGET = ["한겨레", "조선일보", "KBS", "한국일보", "연합뉴스", "JTBC"] #수집할 언론사 목록 작성 혹은 그룹명칭 작성

# ── 수집 방식 On/Off ──────────────────────────────────────────────────────────
USE_QUERY        = True    # True: 검색어로 수집 / False: 해당 언론사 전체 기사
STRICT_FILTER    = True    # True: 제목·본문에 검색어가 실제로 있는 기사만 저장
                           #       (네이버 검색은 유사어·형태소 확장으로 무관 기사를 섞는다)
STRICT_MODE      = "any"   # "any"  : 검색어의 어절 중 하나라도 있으면 통과 (권장)
                           # "exact": 검색어 문자열이 통째로 있어야 통과
                           #   ※ "민주당 전당대회" 같은 다어절 검색어에 exact 를 쓰면
                           #     "민주당 8·17 전당대회" 표기를 놓친다. 기본을 any 로 둔 이유다.
SAVE_RAW_HTML    = True    # 기사 원본 HTML 보존 (재파싱·검증용)
DETECT_GENRE     = True    # 장르 추정 (스트레이트/해설/인터뷰/사설/칼럼)
SEPARATE_IMG_ROWS= True   # True: 이미지 1개당 1행 / False: | 로 이어 한 행
COLLECT_IMAGE    = True
IMAGE_QUALITY    = "low"   # high(원본) / medium(w860) / low(w647)

# ── 댓글 On/Off ───────────────────────────────────────────────────────────────
COLLECT_COMMENTS = True    # 댓글 수집
COMMENT_ACTION   = True    # 댓글 조치유형(삭제 사유) 판정
ACTION_HTML      = "sample"  # "off" | "sample" | "all"  — 위 설명 참조
ACTION_SAMPLE_N  = 30      # sample 모드에서 HTML 로 확인할 기사 수
COMMENT_SORT     = "OLD"   # OLD(등록순) / NEW / FAVORITE
COMMENT_PAGE_MAX = 200
COMMENT_STALL_MAX= 12      # cbox 는 page 파라미터가 불규칙해 중복 페이지를 끼워 넣는다.
                           # 신규 0인 페이지를 몇 번까지 참을지. 3으로 낮추면 100건에서 잘린다.

# ── 병렬·지연 ────────────────────────────────────────────────────────────────
LINK_DELAY     = 0.7
FETCH_WORKERS  = 5
FETCH_DELAY    = 0.15
COMMENT_WORKERS= 5
COMMENT_GAP    = 0.05
HTML_WORKERS   = 3         # Selenium 동시 실행 수. 올리면 차단 위험
MAX_PAGES      = 100 #1페이지 당 기사 10개 이므로, 장기간, 종단 분석 시 Max_pages를 크게 잡기.

ENC = "utf-8-sig"          # 윈도우 엑셀에서 한글 안 깨짐

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
HEADERS = {"User-Agent": UA, "Accept-Language": "ko-KR,ko;q=0.9",
           "Referer": "https://search.naver.com/"}

### 언론사 목록 · 조치유형 사전 · 공통 함수

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  네이버 제휴 언론사 (고유번호 = 1000 + oid)
# ══════════════════════════════════════════════════════════════════════════════
#  확인법: 기사 URL 의 .../article/023/0003989823 에서 023 이 oid → 1023
#  아래 번호는 원 크롤러(네이버뉴스크롤러_27개언론사)에서 URL 직접 확인으로
#  검증된 값이다. 그 과정에서 바로잡힌 것들:
#     프레시안    1143 → 1002 (1143은 오류)
#     아이뉴스24  1109 → 1031 (1109는 OSEN)
#     한겨레21    1227 → 1036 (1227은 아시아경제)
#     주간동아    1020 → 1037 (1020은 동아일보)
#     시사저널    1308 → 1586 (1308은 시사IN)
#     주간경향    1307 → 1033
#     매일신문    1093 → 1088
#     제주일보    1107 → 1084
#     스포츠서울  1076 → 1468 (1076은 스포츠조선)
#  지역지·주간지는 개편이 잦으니 대량 수집 전 한 곳·하루로 시범 실행해 볼 것.
MEDIA_GROUPS = {
    "방송": {
        "KBS": 1056, "MBC": 1214, "SBS": 1055, "YTN": 1052, "연합뉴스TV": 1422,
        "MBN": 1019, "TV조선": 1448, "채널A": 1449, "JTBC": 1437,
    },
    "종합일간지": {
        "경향신문": 1032, "국민일보": 1005, "동아일보": 1020, "문화일보": 1021,
        "서울신문": 1081, "세계일보": 1022, "조선일보": 1023, "중앙일보": 1025,
        "한겨레": 1028, "한국일보": 1469,
    },
    "경제지": {
        "매일경제": 1009, "머니투데이": 1008, "서울경제": 1011, "아시아경제": 1277,
        "이데일리": 1018, "파이낸셜뉴스": 1014, "한국경제": 1015, "한국경제TV": 1004,
        "헤럴드경제": 1016, "조선비즈": 1366, "한경비즈니스": 1050, "SBS Biz": 1374,
        "비즈워치": 1648, "매경이코노미": 1024, "이코노미스트": 1243, "코리아헤럴드": 1044,
    },
    "통신·인터넷": {
        "연합뉴스": 1001, "뉴시스": 1003, "뉴스1": 1421, "오마이뉴스": 1047,
        "프레시안": 1002, "미디어오늘": 1006, "노컷뉴스": 1079, "데일리안": 1119,
        "아이뉴스24": 1031, "뉴스타파": 1607, "더팩트": 1629,
    },
    "주간지·잡지": {
        "한겨레21": 1036, "주간경향": 1033, "주간동아": 1037, "주간조선": 1053,
        "시사저널": 1586, "시사IN": 1308, "신동아": 1262,
    },
    "지역": {
        "부산일보": 1082, "국제신문": 1658, "경기일보": 1666, "강원도민일보": 1654,
        "강원일보": 1087, "대전일보": 1656, "매일신문": 1088, "제주일보": 1084,
        "kbc광주방송": 1660, "CJB청주방송": 1655, "전주MBC": 1659, "대구MBC": 1657,
    },
    "IT·전문": {
        "지디넷코리아": 1092, "전자신문": 1030, "디지털타임스": 1029, "블로터": 1293,
        "디지털데일리": 1138, "농민신문": 1662, "코메디닷컴": 1296, "헬스조선": 1346,
    },
    "스포츠·연예": {
        "스포츠조선": 1076, "스포츠동아": 1382, "스포츠서울": 1468, "일간스포츠": 1241,
        "OSEN": 1109, "스타뉴스": 1108,
    },
}
ALL_MEDIA = {n: v for g in MEDIA_GROUPS.values() for n, v in g.items()}


def resolve_media(target):
    """TARGET(이름 목록 / 그룹명 / 'ALL') → {언론사명: 고유번호}"""
    if isinstance(target, str):
        target = [target]
    out = {}
    for t in target:
        if t == "ALL":
            out.update(ALL_MEDIA)
        elif t in MEDIA_GROUPS:
            out.update(MEDIA_GROUPS[t])
        elif t in ALL_MEDIA:
            out[t] = ALL_MEDIA[t]
        else:
            near = [n for n in ALL_MEDIA if t in n or n in t]
            raise KeyError(f"알 수 없는 언론사/그룹: {t!r}"
                           + (f"  혹시 이것? {near}" if near else
                              f"  (그룹: {list(MEDIA_GROUPS)})"))
    return out


MEDIA = resolve_media(TARGET)


# ══════════════════════════════════════════════════════════════════════════════
#  댓글 조치유형 사전
# ══════════════════════════════════════════════════════════════════════════════
ACTION_TEXT = {
    "작성자에 의해 삭제된 댓글입니다.":                             "작성자삭제",
    "운영규정 미준수로 인해 삭제된 댓글입니다.":                     "운영규정미준수삭제",
    "정보통신망법에 따른 권리침해 요청이 있어, 게시중단 되었습니다.": "권리침해게시중단",
    "클린봇이 부적절한 표현을 감지한 댓글입니다.":                   "클린봇",
}
TEXT_OF = {v: k for k, v in ACTION_TEXT.items()}
STATUS_MAP = {"1": "작성자삭제", "3": "운영규정미준수삭제"}

GENRE_NAME = {1: "스트레이트", 2: "해설/분석", 3: "인터뷰", 4: "사설", 5: "칼럼/기고", 6: "기타"}
COLUMN_MARKS = ["칼럼", "기고", "시론", "발언대", "특별기고", "태평로", "동서남북", "에스프레소",
                "아침햇발", "세상읽기", "유레카", "만물상", "서초포럼", "뉴스룸에서", "현장에서",
                "朝鮮칼럼", "편집국에서", "슬기로운 기자생활"]
ANALYSIS_MARKS = ["뷰리핑", "뉴스 다이브", "논썰", "뉴스 저격", "분석", "팩트체크", "해설",
                  "이슈 인사이드", "심층", "흑백여의도"]
INTERVIEW_MARKS = ["인터뷰", "일문일답", "대담"]


def D(*a):
    p = os.path.join(OUT_DIR, *a)
    os.makedirs(os.path.dirname(p) if os.path.splitext(p)[1] else p, exist_ok=True)
    return p


def clean(s):
    return re.sub(r"[ \t\xa0\u200b]+", " ", str(s or "")).strip()


def flat(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()


def article_key(url):
    m = re.search(r"/article/(?:comment/)?(\d{3,4})/(\d{7,11})", str(url))
    return f"{m.group(1)}/{m.group(2)}" if m else None

## [1] 링크 수집

네이버 뉴스 검색 결과는 requests 로 파싱된다. Selenium 이 필요 없다.

- `USE_QUERY=True` — 검색어별로 수집한 뒤 합집합, 중복 제거
- `USE_QUERY=False` — 해당 언론사 기간 내 전체 기사

네이버는 마지막 페이지를 넘기면 같은 결과를 반복하므로, 신규 0이 2회 연속이면 멈춘다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  1단계 — 링크 수집 (Selenium 불필요)
# ══════════════════════════════════════════════════════════════════════════════
def collect_links_by_query(query, office_id, max_pages=MAX_PAGES):
    oid = f"{office_id - 1000:03d}"
    ds = f"{START_DATE[:4]}.{START_DATE[4:6]}.{START_DATE[6:]}"
    de = f"{END_DATE[:4]}.{END_DATE[4:6]}.{END_DATE[6:]}"
    found, empty = {}, 0
    for page in range(1, max_pages + 1):
        url = ("https://search.naver.com/search.naver?where=news"
               f"&query={query}&start={(page - 1) * 10 + 1}"
               f"&pd=3&ds={ds}&de={de}"
               "&mynews=1&office_type=1&office_section_code=1"
               f"&news_office_checked={office_id}"
               f"&nso=so:r,p:from{START_DATE}to{END_DATE},a:all")
        try:
            r = requests.get(url, headers=HEADERS, timeout=25)
        except Exception:
            time.sleep(3); continue
        soup = BeautifulSoup(r.text, "html.parser")
        new = 0
        for a in soup.select("a[href*='n.news.naver.com']"):
            k = article_key(a.get("href", ""))
            if k and k.startswith(oid + "/") and k not in found:
                found[k] = a["href"].split("?")[0]; new += 1
        # 네이버는 마지막 페이지를 넘기면 같은 결과를 반복한다 → 2회 연속 신규 0이면 종료
        empty = empty + 1 if new == 0 else 0
        if empty >= 2:
            break
        time.sleep(LINK_DELAY)
    return found


def collect_links_all(office_id, max_pages=MAX_PAGES):
    """검색어 없이 해당 언론사 기간 내 전체 기사. USE_QUERY=False 일 때."""
    oid = f"{office_id - 1000:03d}"
    d0 = datetime.strptime(START_DATE, "%Y%m%d")
    d1 = datetime.strptime(END_DATE, "%Y%m%d")
    found = {}
    day = d0
    while day <= d1:
        ymd = day.strftime("%Y%m%d")
        for page in range(1, max_pages + 1):
            url = ("https://news.naver.com/main/list.naver"
                   f"?mode=LPOD&mid=sec&oid={oid}&date={ymd}&page={page}")
            try:
                r = requests.get(url, headers=HEADERS, timeout=25)
            except Exception:
                break
            soup = BeautifulSoup(r.text, "html.parser")
            new = 0
            for a in soup.select("a[href*='article']"):
                k = article_key(a.get("href", ""))
                if k and k.startswith(oid + "/") and k not in found:
                    found[k] = "https://n.news.naver.com/mnews/article/" + k
                    new += 1
            if new == 0:
                break
            time.sleep(LINK_DELAY)
        day += timedelta(days=1)
    return found


def stage_links():
    print("\n[1] 링크 수집")
    union = {}
    for mname, oid in MEDIA.items():
        if USE_QUERY:
            for q in QUERIES:
                got = collect_links_by_query(q, oid)
                print(f"   {mname:8s} | {q:14s} → {len(got):4d}건")
                for k, u in got.items():
                    rec = union.setdefault(k, {"media": mname, "url": u, "keywords": []})
                    rec["keywords"].append(q)
        else:
            got = collect_links_all(oid)
            print(f"   {mname:8s} | 전체기사 → {len(got):4d}건")
            for k, u in got.items():
                union.setdefault(k, {"media": mname, "url": u, "keywords": ["전체기사"]})
    for m in MEDIA:
        print(f"   ├ {m} 합집합 {sum(1 for v in union.values() if v['media'] == m)}건")
    print(f"   └ 전체 {len(union)}건")
    with open(D("_links.json"), "w", encoding="utf-8") as f:
        json.dump(union, f, ensure_ascii=False, indent=1)
    return union

In [ ]:
union = stage_links()
len(union)

## [2] 기사 본문

**핵심.** 네이버는 부제(`strong.media_end_summary`)와 사진설명(`span.end_photo_org`)을
`article#dic_area` **안에** 넣어 둔다. 분리하지 않으면 본문이 오염된다.

| 열 | 내용 |
|---|---|
| `title` | 제목 |
| `subtitle` | 부제 |
| `captions` | 사진설명 |
| `subheads` | 중간제목 |
| `fulltext` | 위 넷을 걷어낸 본문. 문단 구조 보존 |
| `text` | `fulltext` 의 줄바꿈 없는 판본 |

날짜는 표시문자열("2026.07.24. 오전 10:48") 대신 `data-date-time` 속성을 쓴다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  2단계 — 기사 본문 파싱
# ══════════════════════════════════════════════════════════════════════════════
def adjust_image(url, quality="low"):
    if not url or "imgnews.pstatic.net" not in url:
        return url
    from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
    try:
        p = urlparse(url); q = parse_qs(p.query)
        if quality == "high": q.pop("type", None)
        elif quality == "medium": q["type"] = ["w860"]
        else: q["type"] = ["w647"]
        return urlunparse(p._replace(query=urlencode(q, doseq=True)))
    except Exception:
        return url


def detect_genre(title, section, author):
    br = " ".join(re.findall(r"[\[\(【]([^\]\)】]{1,16})[\]\)】]", title))
    if "사설" in br: return 4, "제목[사설]"
    if any(k in br for k in COLUMN_MARKS) or "칼럼" in title: return 5, "제목 칼럼마커"
    if any(k in author for k in ["논설위원", "논설실장", "주필", "편집인", "에디터"]):
        return 5, "바이라인 논설"
    if author and "기자" not in author and "특파원" not in author and len(author) > 2:
        return 5, "바이라인 외부필자"
    if any(k in br for k in INTERVIEW_MARKS) or "인터뷰" in title: return 3, "제목 인터뷰"
    if section == "오피니언": return 5, "섹션 오피니언"
    if any(k in br for k in ANALYSIS_MARKS): return 2, "제목 분석마커"
    return 1, "기본값"


def parse_article(media, url, page_html):
    """★ 부제·사진설명·중간제목을 본문에서 분리한다. 기존 크롤러의 최대 약점."""
    s = BeautifulSoup(page_html, "html.parser")
    d = {"media": media, "url": url}

    t = s.select_one("h2#title_area span") or s.select_one("h2#title_area")
    d["title"] = clean(t.get_text(" ", strip=True)) if t else ""

    dt = s.select_one("span._ARTICLE_DATE_TIME")
    raw = dt.get("data-date-time", "") if dt else ""          # 예: "2026-08-17 10:55:38"
    
    # (내부 로직 유지를 위해 원본 date/time은 남겨둡니다)
    d["date"] = raw[:10]
    d["time"] = raw[11:16] if len(raw) >= 16 else ""
    
    # ★ 추가: ISO 8601 포맷의 datetime 결합 생성
    if raw and len(raw) >= 19:
        d["datetime"] = f"{raw[:10]}T{raw[11:19]}+0900"
    else:
        d["datetime"] = ""

    md = s.select_one("span._ARTICLE_MODIFY_DATE_TIME")
    raw_mod = md.get("data-modify-date-time", "") if md else ""
    
    # ★ 추가: ISO 8601 포맷으로 수정시간 결합
    if raw_mod and len(raw_mod) >= 19:
        d["modify_datetime"] = f"{raw_mod[:10]}T{raw_mod[11:19]}+0900"
    else:
        d["modify_datetime"] = ""
        
    by = s.select_one("span.byline_s")
    d["author"] = clean(by.get_text(" ", strip=True)) if by else ""
    sec = s.select_one("em.media_end_categorize_item")
    d["section"] = clean(sec.get_text(" ", strip=True)) if sec else ""
    d["sections"] = "|".join(clean(e.get_text(" ", strip=True))
                             for e in s.select("em.media_end_categorize_item"))

    m = re.search(r"/article/(\d+)/(\d+)", url)
    d["naver_oid"], d["naver_aid"] = (m.group(1), m.group(2)) if m else ("", "")
    d["pick"] = 1 if s.select_one("i.media_end_head_channel_pick") else 0

    page = ""
    for sp in s.select("span.sds-comps-text-weight-sm"):
        x = clean(sp.get_text())
        if "면" in x: page = x; break
    d["page"] = page                                # 지면·단수 = 의제 중요도 지표

    og = s.select_one("meta[property='og:description']")
    d["og_desc"] = clean(og.get("content", "")) if og else ""
    ogi = s.select_one("meta[property='og:image']")
    d["thumb_url"] = clean(ogi.get("content", "")) if ogi else ""

    cc = s.select_one("span._COMMENT_COUNT_VIEW")
    v = clean(cc.get_text()).replace(",", "") if cc else ""
    d["comment_n"] = int(v) if v.isdigit() else 0
    rt = s.select_one("span.u_likeit_text._count")
    v = clean(rt.get_text()).replace(",", "") if rt else ""
    d["react_total"] = int(v) if v.isdigit() else 0
    for li in s.select("ul._faceLayer li.u_likeit_list"):
        b, c = li.select_one("a._button"), li.select_one("span._count")
        if b and c:
            v = clean(c.get_text()).replace(",", "")
            d[f"react_{b.get('data-type','')}"] = int(v) if v.isdigit() else 0

    d["ai_summary"] = 1 if s.select_one("div#_SUMMARY_BUTTON") else 0
    d["has_origin"] = 1 if s.select_one("a.media_end_head_origin_link") else 0
    d["outlink_n"] = len(s.select("ul.media_end_linked_list > li"))
    iss = [clean(e.get_text()) for e in s.select("strong.related_issue_subject_name")]
    d["issue_n"], d["issues"] = len(iss), "|".join(iss)

    # ── 본문 영역 분해 ────────────────────────────────────────────────────
    d["subtitle"] = d["captions"] = d["subheads"] = d["fulltext"] = ""
    imgs = []
    art = s.select_one("article#dic_area")
    if art:
        a = BeautifulSoup(str(art), "html.parser").select_one("article")
        if COLLECT_IMAGE:
            imgs = [adjust_image(i.get("data-src") or i.get("src") or "", IMAGE_QUALITY)
                    for i in a.select("img")]
            imgs = [x for x in imgs if x]
        d["img_n"] = len(a.select("img"))

        subs = []
        # 1. 클래스가 명시된 공식 부제 추출
        for e in a.select("strong.media_end_summary, div.media_end_summary, h3.media_end_summary"):
            subs.append(clean(e.get_text(" ", strip=True))); e.decompose()

        # 2. ★ 추가: class가 없는 가짜 부제 (본문 최상단의 b, strong 태그) 추출 및 제거
        for e in a.find_all(['b', 'strong']):
            txt = clean(e.get_text(" ", strip=True))
            if not txt or len(txt) < 5: 
                continue # 너무 짧은 단어는 무시
            
            # (A) 본문 시작 부근인지 확인 (이 태그 앞의 텍스트가 40자 이내인지 계산)
            prev_len = 0
            for el in a.descendants:
                if el == e: break
                if isinstance(el, str):
                    prev_len += len(el.strip())
            
            if prev_len > 40:
                continue # 기사 중간에 등장하는 강조 텍스트면 패스
                
            # (B) 인라인 여부 확인 (본문의 일부인지, 독립된 부제인지)
            is_inline = False
            for el in e.next_elements:
                if el in e.descendants: 
                    continue # 자기 자신 내부의 태그는 건너뜀
                
                if isinstance(el, str):
                    if el.strip():
                        is_inline = True # 바로 텍스트가 이어지면 본문의 일부로 간주
                        break
                elif getattr(el, 'name', None) in ['br', 'p', 'div', 'article']:
                    break # 텍스트가 나오기 전에 줄바꿈(<br>)을 만나면 독립된 부제가 맞음
                    
            # 조건에 부합하면 부제로 편입하고 본문에서 삭제
            if not is_inline:
                subs.append(txt)
                e.decompose()

        d["subtitle"] = " | ".join(x for x in subs if x)

        caps = []
        for e in a.select("span.end_photo_org, em.img_desc, div.nbd_table"):
            c = clean(e.get_text(" ", strip=True))
            if c: caps.append(c)
            e.decompose()
        d["captions"] = " || ".join(dict.fromkeys(caps))

        heads = [clean(e.get_text(" ", strip=True)) for e in a.select("strong, b, h3, h4")]
        d["subheads"] = " || ".join(dict.fromkeys(h for h in heads if h and len(h) < 120))

        for e in a.select("script, style"): e.decompose()
        for br in a.select("br"): br.replace_with("\n")
        txt = re.sub(r"[ \t\xa0\u200b]+", " ", a.get_text(""))
        txt = "\n".join(l.strip() for l in txt.split("\n"))
        d["fulltext"] = re.sub(r"\n{3,}", "\n\n", txt).strip()
    else:
        d["img_n"] = 0

    d["images"] = "|".join(imgs)
    d["text"] = flat(d["fulltext"])                 # 줄바꿈 없는 판본
    d["text_len"] = len(d["fulltext"])
    d["fulltext_available"] = 1 if d["text_len"] >= 50 else 0
    if DETECT_GENRE:
        g, why = detect_genre(d["title"], d["section"], d["author"])
        d["genre"], d["genre_label"], d["genre_rule"] = g, GENRE_NAME[g], why
    return d


def fetch_article(job):
    key, media, url = job
    try:
        r = requests.get(url, headers=HEADERS, timeout=25); r.encoding = "utf-8"
        if r.status_code != 200:
            return {"media": media, "url": url, "_key": key, "title": "", "fulltext": "",
                    "fulltext_available": 0, "_error": f"HTTP {r.status_code}"}
        if SAVE_RAW_HTML:
            p = D("01_RAW", media.replace("/", "_"))
            with open(os.path.join(p, f"{key.replace('/', '_')}.html"), "w", encoding="utf-8") as f:
                f.write(r.text)
        d = parse_article(media, url, r.text)
        d["_key"] = key; d["_error"] = ""
        time.sleep(FETCH_DELAY)
        return d
    except Exception as e:
        return {"media": media, "url": url, "_key": key, "title": "", "fulltext": "",
                "fulltext_available": 0, "_error": str(e)[:120]}


def stage_articles(union):
    print("\n[2] 기사 본문 수집")
    jobs = [(k, v["media"], v["url"]) for k, v in union.items()]
    out = []
    with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as ex:
        futs = [ex.submit(fetch_article, j) for j in jobs]
        for f in tqdm(as_completed(futs), total=len(futs), desc="   기사"):
            d = f.result()
            d["search_keyword"] = "|".join(union[d["_key"]]["keywords"])
            out.append(d)
    ok = sum(1 for x in out if x.get("fulltext_available") == 1)
    print(f"   수집 {len(out)}건 / 본문확보 {ok}건")
    return out

In [ ]:
rows = stage_articles(union)
import pandas as pd
pd.DataFrame(rows)[['title','subtitle','captions','date','time','genre_label']].head()

## [3] 엄격 수집 · 중복군

**엄격 수집** — 네이버 검색은 형태소 확장으로 무관 기사를 섞는다. 실제 포함 여부로 거른다.

- `STRICT_MODE="any"` — 검색어의 어절 중 하나라도 있으면 통과 (기본)
- `STRICT_MODE="exact"` — 검색어가 통째로 있어야 통과

`"민주당 전당대회"` 로 검색했는데 기사가 `"민주당 8·17 전당대회"` 로 쓰면 exact 는 놓친다.
그래서 any 를 기본으로 뒀다.

**중복군** — 삭제하지 않고 `duplicate_group` 으로 표시만 한다.
동일기사·업데이트·후속기사 구분은 연구자 몫이다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  3단계 — 엄격 수집 필터
# ══════════════════════════════════════════════════════════════════════════════
def apply_strict_filter(rows):
    """네이버 검색은 형태소 확장으로 무관 기사를 섞는다. 실제 포함 여부로 거른다."""
    if not (STRICT_FILTER and USE_QUERY):
        for r in rows: r["strict_pass"], r["strict_reason"] = 1, "필터 미적용"
        return rows
    kept = 0
    for r in rows:
        hay = f"{r.get('title','')} {r.get('subtitle','')} {r.get('fulltext','')}"
        hit = []
        for q in str(r.get("search_keyword", "")).split("|"):
            q = q.strip().strip('"')
            if not q: continue
            if STRICT_MODE == "exact":
                if q in hay: hit.append(q)
            else:
                if any(w in hay for w in q.split()): hit.append(q)
        r["strict_pass"] = 1 if hit else 0
        r["strict_reason"] = ("포함: " + ", ".join(hit)) if hit else f"검색어 미포함({STRICT_MODE})"
        kept += r["strict_pass"]
    print(f"\n[3] 엄격 수집 [{STRICT_MODE}]: {len(rows)}건 → {kept}건 통과 "
          f"({len(rows)-kept}건 제외)")
    return rows


def assign_duplicate_group(rows):
    """중복은 삭제하지 않고 표시만 한다. 동일기사/업데이트/후속 판단은 연구자 몫."""
    def sig(x):
        t = re.sub(r"[^\w가-힣]", "", unicodedata.normalize("NFKC", x.get("title", "")))
        b = re.sub(r"[^\w가-힣]", "", x.get("fulltext", ""))[:250]
        return hashlib.md5((t[:28] + b).encode()).hexdigest()[:12]
    buckets = {}
    for x in rows: buckets.setdefault(sig(x), []).append(x)
    n = 0
    for _, g in sorted(buckets.items()):
        if len(g) > 1:
            n += 1
            for x in g: x["duplicate_group"] = f"D{n:03d}"
        else:
            g[0]["duplicate_group"] = ""
    print(f"   중복군 {n}개 (삭제하지 않고 duplicate_group 표시)")
    return n

In [ ]:
rows = apply_strict_filter(rows)
assign_duplicate_group(rows)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  4단계 — 댓글 수집 (JSON API)
# ══════════════════════════════════════════════════════════════════════════════
_tl = threading.local(); _lock = threading.Lock(); _last = [0.0]
CBOX = "https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json"


def _session(ref):
    if not hasattr(_tl, "s"):
        s = requests.Session()
        s.mount("https://", requests.adapters.HTTPAdapter(pool_connections=16, pool_maxsize=16))
        s.headers.update({"User-Agent": UA, "X-Requested-With": "XMLHttpRequest",
                          "Accept": "application/json, text/javascript, */*; q=0.01",
                          "Accept-Language": "ko-KR,ko;q=0.9", "Connection": "keep-alive"})
        _tl.s = s
    _tl.s.headers["Referer"] = ref
    return _tl.s


def _throttle():
    with _lock:
        w = COMMENT_GAP - (time.time() - _last[0])
        if w > 0: time.sleep(w)
        _last[0] = time.time()


def fetch_comments(key, url):
    """includeAllStatus=true 로 삭제·클린봇 건까지 받는다."""
    oid = article_key(url)
    if not oid:
        return {"_key": key, "ok": False, "error": "oid없음", "comments": []}
    ref = f"https://n.news.naver.com/mnews/article/comment/{oid}"
    seen, out, stall = set(), [], 0
    
    # 다음 페이지 요청을 위한 변수 초기화
    next_param = ""
    
    for page in range(1, COMMENT_PAGE_MAX + 1):
        _throttle()
        p = {"ticket": "news", "templateId": "default_society", "pool": "cbox5",
             "lang": "ko", "country": "KR", "objectId": f"news{oid.replace('/', ',')}",
             "pageSize": 100, "indexSize": 10, "page": page, "sort": COMMENT_SORT,
             "includeAllStatus": "true", "_callback": "cb"}
        
        # 2페이지 이상일 경우 (next_param 값이 존재할 때) 추가 파라미터 삽입
        if next_param:
            p["pageType"] = "more"
            p["moreParam.next"] = next_param

        try:
            t = _session(ref).get(CBOX, params=p, timeout=15).text
            js = json.loads(t[t.index("(") + 1: t.rindex(")")])
        except Exception as e:
            return {"_key": key, "ok": False, "error": str(e)[:120],
                    "comments": out, "n_pages": page}
        
        res = js.get("result") or {}
        cl = res.get("commentList") or []
        
        # API 응답에서 다음 페이지를 위한 next 값 추출 후 갱신
        next_param = (res.get("morePage") or {}).get("next", "")
        
        new = 0
        for c in cl:
            no = str(c.get("commentNo", ""))
            if no and no not in seen:
                seen.add(no); out.append(c); new += 1
                
        if not cl: break
        
        stall = stall + 1 if new == 0 else 0
        if stall >= COMMENT_STALL_MAX: break
        
        # 다음 페이지 값이 비어있다면 더 이상 수집할 댓글이 없는 것이므로 루프 종료
        if not next_param:
            break
            
    return {"_key": key, "ok": True, "error": "", "comments": out}


def judge_action(c, html_text=None):
    """조치유형 판정. HTML 문구가 있으면 그것이 정답, 없으면 JSON status."""
    if html_text and html_text in ACTION_TEXT:
        return ACTION_TEXT[html_text], "HTML문구"
    if c.get("hiddenByCleanbot"):
        return "클린봇", "JSON_cleanbot"
    if not c.get("deleted"):
        return "노출", "JSON_정상"
    st = str(c.get("status"))
    if st in STATUS_MAP:
        return STATUS_MAP[st], f"JSON_status{st}"
    if c.get("inspectionId"):
        return "권리침해게시중단", "JSON_inspectionId"
    return "미분류", f"status{st}_미관측"


def stage_comments(rows, union):
    if not COLLECT_COMMENTS:
        return []
    print("\n[4] 댓글 수집")
    targets = [(r["_key"], r["url"]) for r in rows
               if r.get("strict_pass", 1) == 1 and not r.get("_error")]
    ck = D("_comments_ckpt.jsonl")
    done = set()
    if os.path.exists(ck):
        for l in open(ck, encoding="utf-8"):
            try: done.add(json.loads(l)["_key"])
            except Exception: pass
        print(f"   체크포인트 {len(done)}건 건너뜀")
    todo = [t for t in targets if t[0] not in done]

    with open(ck, "a", encoding="utf-8") as f:
        with ThreadPoolExecutor(max_workers=COMMENT_WORKERS) as ex:
            futs = [ex.submit(fetch_comments, k, u) for k, u in todo]
            for fu in tqdm(as_completed(futs), total=len(futs), desc="   댓글"):
                res = fu.result()
                with _lock:
                    f.write(json.dumps(res, ensure_ascii=False) + "\n"); f.flush()

    arts = []
    for l in open(ck, encoding="utf-8"):
        try: arts.append(json.loads(l))
        except Exception: pass
    n = sum(len(a.get("comments", [])) for a in arts)
    print(f"   기사 {len(arts)}건 / 댓글 {n:,}건")
    return arts

In [ ]:
arts = stage_comments(rows, union)
sum(len(a.get('comments', [])) for a in arts)

## [5] HTML 대체문구 검증 (선택)

JSON 응답에는 화면에 뜨는 대체문구가 **단 한 글자도 없다.** status 코드만 준다.
반대로 댓글 페이지 HTML 에는 문구가 있으나 JS 렌더링이라 requests 로는 못 받는다.

- `ACTION_HTML="off"` — JSON 만. 빠르다
- `ACTION_HTML="sample"` — 표본만 읽어 status↔문구 매핑 대조 (권장)
- `ACTION_HTML="all"` — 전 기사. 매우 느리다

교차표에서 각 status 가 문구 1종에만 대응하면 `all` 을 돌릴 필요가 없다.

**`display:none` 제외가 중요하다.** 클린봇 블록 안에는 숨겨진 '차단한 이용자' 문구가 있어
그냥 읽으면 오판한다.

selenium 이 없으면 이 셀은 건너뛰고 JSON 판정만 쓴다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  5단계 — HTML 대체문구 검증 (선택, Selenium)
# ══════════════════════════════════════════════════════════════════════════════
def stage_action_html(arts, union):
    """JSON status ↔ 실제 대체문구 매핑이 맞는지 HTML 로 대조한다."""
    if not COMMENT_ACTION or ACTION_HTML == "off" or not arts:
        return {}
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
        from selenium.webdriver.support import expected_conditions as EC
        import chromedriver_autoinstaller, ssl
    except ImportError:
        print("\n[5] selenium 미설치 — HTML 검증 건너뜀 "
              "(pip install selenium chromedriver-autoinstaller)")
        return {}

    print(f"\n[5] HTML 대체문구 검증 [{ACTION_HTML}]")
    ssl._create_default_https_context = ssl._create_unverified_context
    chromedriver_autoinstaller.install()
    dtl = threading.local(); drivers = []; dlock = threading.Lock()

    def drv():
        if not hasattr(dtl, "d"):
            o = Options()
            for a in ["--headless=new", "--no-sandbox", "--disable-dev-shm-usage",
                      "--disable-gpu", "--blink-settings=imagesEnabled=false", "lang=ko_KR"]:
                o.add_argument(a)
            o.page_load_strategy = "eager"
            d = webdriver.Chrome(options=o); d.set_page_load_timeout(40)
            dtl.d = d
            with dlock: drivers.append(d)
        return dtl.d

    def visible(tag):
        """★ 클린봇 블록 안에는 display:none 인 '차단한 이용자' 문구가 숨어 있다.
           걸러내지 않으면 오판한다."""
        return tag and "display:none" not in (tag.get("style") or "").replace(" ", "")

    # data-info 안의 값들. 화면 표시가 아니라 원본 값이다.
    RE_NO  = re.compile(r"commentNo\s*:\s*'(\d+)'")
    RE_REG = re.compile(r"regTime\s*:\s*'([^']+)'")     # ★ 진짜 작성시각
    RE_DEL = re.compile(r"deleted\s*:\s*(true|false)")

    # 보이는 비노출 문구 개수를 브라우저 안에서 센다.
    # page_source 를 매번 내려받아 파싱하는 것보다 훨씬 싸다.
    JS_COUNT = ("return Array.from(document.querySelectorAll("
                "'.u_cbox_delete_contents,.u_cbox_cleanbot_contents'))"
                ".filter(function(e){return e.offsetParent!==null;}).length;")

    def read_article(key, url, need):
        oid = article_key(url)
        d = drv()
        try:
            d.get(f"https://n.news.naver.com/mnews/article/comment/{oid}")
            WebDriverWait(d, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "li.u_cbox_comment")))
        except Exception as e:
            return {"_key": key, "rows": [], "clicks": 0, "error": str(e)[:80]}

        clicks = 0
        for i in range(60):                                # 더보기 — 목표만큼 모이면 중단
            try:
                if need and d.execute_script(JS_COUNT) >= need: break
            except Exception:
                pass
            try:
                b = d.find_element(By.CSS_SELECTOR, "a.u_cbox_btn_more")
                if not b.is_displayed(): break
                d.execute_script("arguments[0].click();", b)
                clicks = i + 1
                time.sleep(0.4)
            except Exception:
                break

        rows = []
        for li in BeautifulSoup(d.page_source, "html.parser").select("li.u_cbox_comment"):
            txt = ""
            for t in (li.select_one("span.u_cbox_cleanbot_contents"),
                      li.select_one("span.u_cbox_delete_contents")):
                if visible(t) and t.get_text(strip=True) in ACTION_TEXT:
                    txt = t.get_text(strip=True); break
            if not txt: continue
            info = li.get("data-info") or ""
            mno, mreg, mdel = RE_NO.search(info), RE_REG.search(info), RE_DEL.search(info)
            dt = li.select_one("span.u_cbox_date")
            rows.append({
                "commentNo": mno.group(1) if mno else "",
                "reg_time":  mreg.group(1) if mreg else "",          # 작성시각
                # ★ 화면에 뜨는 이 값은 조치시각이다. 작성시각으로 쓰면 안 된다.
                "shown_time": (dt.get("data-value") if dt else "") or "",
                "deleted":   bool(mdel and mdel.group(1) == "true"),
                "문구": txt,
            })
        return {"_key": key, "rows": rows, "clicks": clicks, "error": ""}

    need_by = {a["_key"]: sum(1 for c in a.get("comments", [])
                              if c.get("deleted") or c.get("hiddenByCleanbot"))
               for a in arts}
    cand = [(a["_key"], union[a["_key"]]["url"]) for a in arts if need_by.get(a["_key"], 0) > 0]
    if ACTION_HTML == "sample":
        cand = sorted(cand, key=lambda x: -need_by[x[0]])[:ACTION_SAMPLE_N]
    print(f"   대상 {len(cand)}기사")

    ck = D("_action_html_ckpt.jsonl")
    done = set()
    if os.path.exists(ck):
        for l in open(ck, encoding="utf-8"):
            try: done.add(json.loads(l)["_key"])
            except Exception: pass
    cand = [c for c in cand if c[0] not in done]
    if cand:
        with open(ck, "a", encoding="utf-8") as f:
            with ThreadPoolExecutor(max_workers=HTML_WORKERS) as ex:
                futs = [ex.submit(read_article, k, u, need_by.get(k, 0)) for k, u in cand]
                for fu in tqdm(as_completed(futs), total=len(futs), desc="   HTML"):
                    with _lock:
                        f.write(json.dumps(fu.result(), ensure_ascii=False) + "\n"); f.flush()
    for d in drivers:
        try: d.quit()
        except Exception: pass

    idx = {}
    if os.path.exists(ck):
        for l in open(ck, encoding="utf-8"):
            try:
                a = json.loads(l)
                for r in a.get("rows", []):
                    if r.get("commentNo"): idx[str(r["commentNo"])] = r["문구"]
            except Exception: pass
    print(f"   HTML 문구 확보 {len(idx)}건")
    return idx


def crosscheck_mapping(arts, html_idx):
    """JSON 코드 × HTML 실제 문구 교차표. 1:1 이면 JSON 만으로 충분하다는 근거."""
    if not html_idx: return
    from collections import Counter
    pairs = Counter()
    for a in arts:
        for c in a.get("comments", []):
            no = str(c.get("commentNo", ""))
            if no not in html_idx: continue
            code = ("cleanbot" if c.get("hiddenByCleanbot")
                    else f"status{c.get('status')}" + ("+insp" if c.get("inspectionId") else ""))
            pairs[(code, ACTION_TEXT[html_idx[no]])] += 1
    if not pairs: return
    t = pd.Series(pairs).unstack(fill_value=0)
    print("\n   [JSON 코드] × [HTML 실제 문구]")
    print("   " + t.to_string().replace("\n", "\n   "))
    mixed = [i for i in t.index if (t.loc[i] > 0).sum() > 1]
    print("   " + ("✓ 각 코드가 문구 1종에만 대응 — JSON 매핑 신뢰 가능"
                   if not mixed else f"⚠ 복수 문구로 갈리는 코드: {mixed} → ACTION_HTML='all' 권장"))

In [ ]:
html_idx = stage_action_html(arts, union) if arts else {}
if arts and html_idx:
    crosscheck_mapping(arts, html_idx)

## [6] 저장

### 산출물

```
02_METADATA/articles.csv · .xlsx    기사 1건 = 1행
02_METADATA/comments.csv            댓글 1건 = 1행
03_TEXT/{매체}/{article_id}.txt      제목·부제·중간제목·사진설명·본문
01_RAW/{매체}/*.html                 원본 HTML
04_LOG/collection_log.json
```

### 댓글 표의 시각 열

| 열 | 내용 |
|---|---|
| `reg_time` | **작성시각.** 삭제돼도 보존된다 |
| `action_time` | **조치시각.** 노출 댓글은 빈 칸 |
| `survival_min` / `survival_day` | 작성 후 조치까지 살아있던 시간 |
| `mod_time` | 원본 modTime |
| `cleanbot_original` | 클린봇이 가리기 전 원문 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  6단계 — 저장
# ══════════════════════════════════════════════════════════════════════════════
ART_COLS = ["article_id", "media", "datetime", "modify_datetime", "title", "subtitle", "author",
            "section", "genre", "genre_label", "genre_rule", "url", "text", "fulltext",
            "fulltext_available", "search_keyword", "strict_pass", "strict_reason",
            "duplicate_group", "captions", "subheads", "page", "pick",
            "text_len", "comment_n", "react_total", "og_desc",
            "sections", "img_n", "images", "thumb_url", "issue_n", "issues",
            "outlink_n", "ai_summary", "has_origin", "naver_oid", "naver_aid",
            "txt_file", "raw_html", "_error"]

# reg_time    작성시각   — 삭제돼도 API 가 보존한다
# action_time 조치시각   — 삭제·게시중단이 실행된 시각. 노출 댓글은 빈 칸
# mod_time    원본 modTime (노출 댓글은 reg_time 과 동일)
# survival_min 작성 후 조치까지 몇 분 살아있었나
CMT_COLS = ["comment_id", "article_id", "media", "article_date", "article_title",
            "comment_no", "user_name",
            "reg_time", "action_time", "survival_min", "survival_day", "mod_time",
            "comment_text", "action_type", "action_text", "action_source",
            "cleanbot_original", "is_deleted", "is_cleanbot", "status", "inspection_id",
            "sympathy", "antipathy", "reply_n", "is_best",
            "parent_no", "reply_level", "article_url"]


def stage_save(rows, arts, html_idx):
    print("\n[6] 저장")
    for s in ("01_RAW", "02_METADATA", "03_TEXT", "04_LOG"):
        os.makedirs(os.path.join(OUT_DIR, s), exist_ok=True)

    # article_id 부여
    rows.sort(key=lambda x: (x["media"], x.get("date", ""), x.get("time", "")))
    cnt = {}
    code = {m: re.sub(r"[^A-Za-z가-힣]", "", m)[:6] for m in MEDIA}
    for x in rows:
        c = code.get(x["media"], "MEDIA")
        day = (x.get("date") or "0000-00-00").replace("-", "")
        cnt[(c, day)] = cnt.get((c, day), 0) + 1
        x["article_id"] = f"{c}_{day}_{cnt[(c, day)]:03d}"
        x["txt_file"] = os.path.join("03_TEXT", x["media"], f"{x['article_id']}.txt")
        x["raw_html"] = (os.path.join("01_RAW", x["media"],
                                      f"{x['_key'].replace('/', '_')}.html")
                         if SAVE_RAW_HTML else "")

    # 03_TEXT
    for x in rows:
        p = D("03_TEXT", x["media"])
        with open(os.path.join(p, f"{x['article_id']}.txt"), "w", encoding=ENC) as f:
            # datetime 바로 다음에 modify_datetime 추가
            for k in ("article_id", "media", "datetime", "modify_datetime", "author", "section",
                      "genre_label", "url"):
                f.write(f"{k}: {x.get(k,'')}\n")
            f.write("=" * 70 + "\n")
            f.write(f"[제목] {x.get('title','')}\n\n")
            if x.get("subtitle"):  f.write(f"[부제] {x['subtitle']}\n\n")
            if x.get("subheads"):  f.write(f"[중간제목] {x['subheads']}\n\n")
            if x.get("captions"):  f.write(f"[사진설명] {x['captions']}\n\n")
            f.write("[본문]\n" + x.get("fulltext", ""))

    df = pd.DataFrame([{k: x.get(k, "") for k in ART_COLS} for x in rows])
    if SEPARATE_IMG_ROWS and COLLECT_IMAGE:
        df = df.assign(images=df["images"].str.split("|")).explode("images")
    ap = os.path.join(OUT_DIR, "02_METADATA", "articles.csv")
    df.to_csv(ap, index=False, encoding=ENC, quoting=csv.QUOTE_ALL)
    try: df.to_excel(ap.replace(".csv", ".xlsx"), index=False)
    except Exception as e: print(f"   ⚠ xlsx 저장 실패: {e}")
    print(f"   {ap}  {len(df)}행")

    # ── 댓글 ──────────────────────────────────────────────────────────────
    if arts:
        by_key = {x["_key"]: x for x in rows}
        crows = []
        for a in arts:
            art = by_key.get(a["_key"])
            if not art: continue
            for c in a.get("comments", []):
                no = str(c.get("commentNo", ""))
                atype, src = judge_action(c, html_idx.get(no))
                # regTime 은 삭제돼도 작성시각을 보존한다. 화면에 뜨는 것과 다르다.
                reg, mod = c.get("regTime", ""), c.get("modTime", "")
                acted = bool(c.get("deleted"))
                surv = sday = ""
                if acted and reg and mod:
                    try:
                        surv = round((pd.Timestamp(mod) - pd.Timestamp(reg))
                                     .total_seconds() / 60, 1)
                        sday = round(surv / 1440, 2)
                    except Exception: pass
                body = str(c.get("contents") or "")
                crows.append({
                    "comment_id": f"{art['article_id']}_C{len(crows)+1:05d}",
                    "article_id": art["article_id"], "media": art["media"],
                    "article_date": art.get("date", ""), "article_title": art.get("title", ""),
                    "comment_no": no, "user_name": c.get("userName", ""),
                    "reg_time": reg,                       # 작성시각
                    "action_time": mod if acted else "",   # 조치시각 (노출 댓글은 빈 칸)
                    "survival_min": surv, "survival_day": sday,
                    "mod_time": mod,
                    # 삭제 댓글은 본문이 없으므로 화면에 뜨는 대체문구를 넣는다
                    "comment_text": body if atype in ("노출", "클린봇") else TEXT_OF.get(atype, ""),
                    "action_type": atype, "action_text": TEXT_OF.get(atype, ""),
                    "action_source": src,
                    # ★ 클린봇은 화면에서만 가려질 뿐 원문이 API 에 살아 있다
                    "cleanbot_original": body if atype == "클린봇" else "",
                    "is_deleted": 1 if acted else 0,
                    "is_cleanbot": 1 if c.get("hiddenByCleanbot") else 0,
                    "status": c.get("status", ""), "inspection_id": c.get("inspectionId") or "",
                    "sympathy": c.get("sympathyCount", 0), "antipathy": c.get("antipathyCount", 0),
                    "reply_n": c.get("replyAllCount", 0), "is_best": 1 if c.get("best") else 0,
                    "parent_no": c.get("parentCommentNo", ""), "reply_level": c.get("replyLevel", ""),
                    "article_url": art.get("url", ""),
                })
        cd = pd.DataFrame(crows, columns=CMT_COLS)
        cp = os.path.join(OUT_DIR, "02_METADATA", "comments.csv")
        cd.to_csv(cp, index=False, encoding=ENC, quoting=csv.QUOTE_ALL)
        print(f"   {cp}  {len(cd)}행")

        print("\n   [조치유형]")
        for k, v in cd["action_type"].value_counts().items():
            print(f"     {k:20s} {v:6,}")
        print("   [판정근거]")
        for k, v in cd["action_source"].value_counts().items():
            print(f"     {k:20s} {v:6,}")
        cb = cd[cd.action_type == "클린봇"]
        if len(cb):
            n = (cb["cleanbot_original"].astype(str).str.strip() != "").sum()
            print(f"   [클린봇] {len(cb):,}건 → 가려진 원문 복원 {n:,}건 "
                  f"({n/len(cb)*100:.1f}%)")
        sv = pd.to_numeric(cd["survival_min"], errors="coerce").dropna()
        if len(sv):
            print(f"\n   [생존시간] 작성 → 조치까지  n={len(sv):,}")
            print(f"     중앙 {sv.median():,.0f}분 ({sv.median()/1440:.1f}일) · "
                  f"평균 {sv.mean():,.0f}분")
            print(f"     1시간내 {(sv<60).mean()*100:.1f}% · 1일내 {(sv<1440).mean()*100:.1f}% · "
                  f"7일내 {(sv<10080).mean()*100:.1f}%")
            g = (cd.assign(_s=pd.to_numeric(cd["survival_min"], errors="coerce"))
                   .dropna(subset=["_s"]).groupby("action_type")["_s"]
                   .agg(["count", "median", "mean"]).round(1))
            print("   [조치유형별 생존시간(분)]")
            print("   " + g.to_string().replace("\n", "\n   "))

    # 로그
    with open(os.path.join(OUT_DIR, "04_LOG", "collection_log.json"), "w", encoding="utf-8") as f:
        json.dump({
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "period": f"{START_DATE}~{END_DATE}", "media": list(MEDIA),
            "queries": QUERIES if USE_QUERY else ["(전체기사)"],
            "options": {"USE_QUERY": USE_QUERY, "STRICT_FILTER": STRICT_FILTER,
                        "STRICT_MODE": STRICT_MODE, "SAVE_RAW_HTML": SAVE_RAW_HTML,
                        "DETECT_GENRE": DETECT_GENRE, "COLLECT_IMAGE": COLLECT_IMAGE,
                        "COLLECT_COMMENTS": COLLECT_COMMENTS,
                        "COMMENT_ACTION": COMMENT_ACTION, "ACTION_HTML": ACTION_HTML},
            "n_articles": len(rows),
            "n_strict_pass": sum(x.get("strict_pass", 1) for x in rows),
            "n_comments": sum(len(a.get("comments", [])) for a in arts),
        }, f, ensure_ascii=False, indent=1)
    print(f"\n★ 완료 — {OUT_DIR}")


# ══════════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    os.makedirs(OUT_DIR, exist_ok=True)
    print("=" * 78)
    print(f"  네이버 뉴스 수집  |  {START_DATE}~{END_DATE}  |  {', '.join(MEDIA)}")
    print(f"  검색어: {QUERIES if USE_QUERY else '(전체기사)'}")
    print(f"  엄격수집 {STRICT_FILTER}[{STRICT_MODE}] · 댓글 {COLLECT_COMMENTS} · "
          f"조치유형 {COMMENT_ACTION}[{ACTION_HTML}]")
    print("=" * 78)

    union = stage_links()
    rows = stage_articles(union)
    rows = apply_strict_filter(rows)
    assign_duplicate_group(rows)
    arts = stage_comments(rows, union)
    html_idx = stage_action_html(arts, union) if arts else {}
    if arts and html_idx:
        crosscheck_mapping(arts, html_idx)
    stage_save(rows, arts, html_idx)
    print(f"  소요 {time.time()-t0:.0f}초")

In [ ]:
stage_save(rows, arts, html_idx)

## 한 번에 실행

위 셀들을 순서대로 실행했다면 이 셀은 필요 없다.
처음부터 다시 돌릴 때만 쓴다.

In [ ]:
def main():
    t0 = time.time()
    os.makedirs(OUT_DIR, exist_ok=True)
    print("=" * 78)
    print(f"  네이버 뉴스 수집  |  {START_DATE}~{END_DATE}  |  {', '.join(MEDIA)}")
    print(f"  검색어: {QUERIES if USE_QUERY else '(전체기사)'}")
    print(f"  엄격수집 {STRICT_FILTER}[{STRICT_MODE}] · 댓글 {COLLECT_COMMENTS} · "
          f"조치유형 {COMMENT_ACTION}[{ACTION_HTML}]")
    print("=" * 78)

    union = stage_links()
    rows = stage_articles(union)
    rows = apply_strict_filter(rows)
    assign_duplicate_group(rows)
    arts = stage_comments(rows, union)
    html_idx = stage_action_html(arts, union) if arts else {}
    if arts and html_idx:
        crosscheck_mapping(arts, html_idx)
    stage_save(rows, arts, html_idx)
    print(f"  소요 {time.time()-t0:.0f}초")


# ══════════════════════════════════════════════════════════════════════════════
#  언론사 목록은 파일 상단 MEDIA_GROUPS 에 코드로 들어 있다.
#  목록 확인:  python -c "import naver_news_collector as n; print(n.MEDIA_GROUPS)"
#  그룹 확인:  python -c "import naver_news_collector as n; print(list(n.MEDIA_GROUPS))"
# ══════════════════════════════════════════════════════════════════════════════


# if __name__ == "__main__":
    main()

# main()   # ← 주석을 풀면 전 단계를 한 번에 실행

In [ ]:
for g, d in MEDIA_GROUPS.items():
    print(f'{g:12s} {len(d):2d}곳  ' + ', '.join(d))
print(f'\n전체 {len(ALL_MEDIA)}곳 · 현재 선택 {MEDIA}')